In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'batch_size': 256, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

{'num_examples': 1500, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   

                                  sentence2  label  
0      A man wearing a hard hat is dancing.   5.00  
1                A child is riding a horse.   4.75  
2  The man is feeding a mouse to the snake.   5.00  
3                  A man is playing guitar.   2.40  
4                 A man is playing a flute.   2.75  


In [3]:
model = SentenceTransformer(model_name, device=device)
print(model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers/all-MiniLM-L6-v2


In [4]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

norms1 = np.linalg.norm(emb1, axis=1)
norms2 = np.linalg.norm(emb2, axis=1)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [5]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["embedding_norm_s1"] = norms1
results_df["embedding_norm_s2"] = norms2

compact_table = results_df[[
    "sentence1",
    "sentence2",
    "label",
    "cosine_similarity",
    "predicted_score_0_5",
]].head(10).copy()

print(compact_table)

                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  label  cosine_similarity  \
0      A man wearing a hard hat is dancing.  5.000           0.993372   
1                A child is riding a horse.  4.750           0.953962   
2  The man is feeding a mouse to the snake.  5.000           0.980101   
3                  A man is playing guitar.  2.400           0.647083   
4                 A man is playing a flute.  2.750           0.746228   
5                  A man is cutting onions.  2.615           0.754619   
6       The man

In [6]:
runtime_seconds = time.time() - start_time

norm_summary = pd.DataFrame({
    "sentence1_norm": norms1,
    "sentence2_norm": norms2,
}).agg(["mean", "std", "min", "max"])

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
print("embedding_norm_summary:")
print(norm_summary)

device_used: mps
model_name: sentence-transformers/all-MiniLM-L6-v2
dataset_split: glue/stsb/validation
num_examples: 1500
pearson_correlation: 0.869619
spearman_correlation: 0.867164
runtime_seconds: 8.72
embedding_norm_summary:
      sentence1_norm  sentence2_norm
mean    1.000000e+00    1.000000e+00
std     2.879963e-08    2.847795e-08
min     9.999999e-01    9.999999e-01
max     1.000000e+00    1.000000e+00
